## 兩組新樣本：批次轉換與疊圖比較
此段會分別處理兩組樣本：

1. 第一組（極化）：`WWjj_EW-LL-WW_cmf`、`WWjj_EW-LT-WW_cmf`、`WWjj_EW-TT-WW_cmf`
2. 第二組（EW/QCD）：`WWjj_EW`、`WWjj_QCD`、`WZjj_EW`、`WZjj_QCD`

目前先只測試每個資料夾的 `run_01`；確認無誤後再加入 `run_02`。

每一組皆會：

- 掃描 `*.root`
- 逐檔執行 `export_sr_to_parquet.py` 轉為 SR-only parquet（顯示事件級進度條）
- 生成該組的 manifest
- 依變數輸出同圖疊圖 PDF（`figsize=(5, 4)`）

In [ ]:
import json
import subprocess
from pathlib import Path

repo_root = Path('/home/r10222035/longitudinal_W')
script_path = repo_root / 'Sample/selection/export_sr_to_parquet.py'

target_dirs = [
    repo_root / 'Sample/MG5/WWjj_EW-LL-WW_cmf',
    repo_root / 'Sample/MG5/WWjj_EW-LT-WW_cmf',
    repo_root / 'Sample/MG5/WWjj_EW-TT-WW_cmf',
]

# 先只測試 run_01；確認流程沒問題後再把 run_02 加進來。
target_runs = {'run_01', 'run_02'}

batch_out_dir = repo_root / 'Sample/Parquet/batch_sr_parquet_ew_polar'
batch_out_dir.mkdir(parents=True, exist_ok=True)
manifest_path = batch_out_dir / 'batch_manifest_ew_polar.json'

all_root_files = []
for d in target_dirs:
    roots = sorted(
        p for p in d.rglob('*.root')
        if any(run in p.parts for run in target_runs)
    )
    all_root_files.extend(roots)

assert script_path.exists(), f'Missing script: {script_path}'
assert all_root_files, 'No ROOT files found for selected runs in target directories.'

print(f'[EW polar] Runs: {sorted(target_runs)}')
print(f'[EW polar] Found {len(all_root_files)} ROOT files.')

converted_records = []
for i, root_file in enumerate(all_root_files, start=1):
    sample = root_file.parts[-4] if len(root_file.parts) >= 4 else root_file.parent.name
    run_name = root_file.parent.name

    out_name = f'{sample}_{run_name}_sr.parquet'
    out_parquet = batch_out_dir / out_name

    cmd = [
        'conda', 'run', '-n', 'vbs_wl',
        'python', str(script_path),
        '--input-root', str(root_file),
        '--tree', 'Delphes',
        '--output-parquet', str(out_parquet),
    ]

    print(f'[EW polar {i}/{len(all_root_files)}] sample={sample} converting: {root_file}')
    print(' '.join(cmd))
    proc = subprocess.run(cmd, cwd=str(repo_root))

    if proc.returncode != 0:
        raise RuntimeError(f'Conversion failed for {root_file}')

    cutflow_path = Path(str(out_parquet) + '.cutflow.json')
    assert out_parquet.exists(), f'Missing parquet: {out_parquet}'
    assert cutflow_path.exists(), f'Missing cutflow: {cutflow_path}'

    with cutflow_path.open('r', encoding='utf-8') as f:
        info = json.load(f)

    converted_records.append(
        {
            'sample': sample,
            'run': run_name,
            'input_root': str(root_file),
            'parquet': str(out_parquet),
            'cutflow_json': str(cutflow_path),
            'n_output_rows': int(info.get('n_output_rows', 0)),
            'cutflow': info.get('cutflow', {}),
        }
    )

manifest_path.write_text(json.dumps(converted_records, indent=2), encoding='utf-8')

print(f'[EW polar] Converted {len(converted_records)} files.')
print(f'[EW polar] Manifest: {manifest_path}')
for rec in converted_records:
    print(f"- {rec['sample']} {rec['run']}: rows={rec['n_output_rows']}")

In [1]:
import re
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq

repo_root = Path('/home/r10222035/longitudinal_W')
batch_out_dir = repo_root / 'Sample/Parquet/batch_sr_parquet_ew_polar'
manifest_path = batch_out_dir / 'batch_manifest_ew_polar.json'
overlay_dir = repo_root / 'figures' / 'sr_overlay_ew_polar'
overlay_dir.mkdir(parents=True, exist_ok=True)

assert manifest_path.exists(), f'Missing manifest: {manifest_path}'
records = json.loads(manifest_path.read_text(encoding='utf-8'))
assert records, 'Manifest is empty.'


def sanitize_filename(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', name)


def x_label_math(col: str) -> str:
    label_map = {
        'l1_pt': r'$p_{\mathrm{T}}^{\ell_1}\,[\mathrm{GeV}]$',
        'l2_pt': r'$p_{\mathrm{T}}^{\ell_2}\,[\mathrm{GeV}]$',
        'j1_pt': r'$p_{\mathrm{T}}^{j_1}\,[\mathrm{GeV}]$',
        'j2_pt': r'$p_{\mathrm{T}}^{j_2}\,[\mathrm{GeV}]$',
        'pt_ll': r'$p_{\mathrm{T}}^{\ell\ell}\,[\mathrm{GeV}]$',
        'met_et': r'$E_{\mathrm{T}}^{\mathrm{miss}}\,[\mathrm{GeV}]$',
        'm_ll': r'$m_{\ell\ell}\,[\mathrm{GeV}]$',
        'm_jj': r'$m_{jj}\,[\mathrm{GeV}]$',
        'mt_l1_met': r'$m_{\mathrm{T}}(\ell_1, E_{\mathrm{T}}^{\mathrm{miss}})\,[\mathrm{GeV}]$',
        'mt_l2_met': r'$m_{\mathrm{T}}(\ell_2, E_{\mathrm{T}}^{\mathrm{miss}})\,[\mathrm{GeV}]$',
        'mt_ll_met': r'$m_{\mathrm{T}}(\ell\ell, E_{\mathrm{T}}^{\mathrm{miss}})\,[\mathrm{GeV}]$',
        'mt0_ll_met': r'$m_{\mathrm{T}}^{0}(\ell\ell, E_{\mathrm{T}}^{\mathrm{miss}})\,[\mathrm{GeV}]$',
        'l1_eta': r'$\eta_{\ell_1}$',
        'l2_eta': r'$\eta_{\ell_2}$',
        'j1_eta': r'$\eta_{j_1}$',
        'j2_eta': r'$\eta_{j_2}$',
        'deta_ll': r'$\Delta\eta_{\ell\ell}$',
        'dy_jj': r'$\Delta y_{jj}$',
        'dr_ll': r'$\Delta R_{\ell\ell}$',
        'dr_jj': r'$\Delta R_{jj}$',
        'min_dr_lj': r'$\min(\Delta R_{\ell,j})$',
        'dphi_l2_l1': r'$\Delta\phi(\ell_2,\ell_1)$',
        'dphi_j1_l1': r'$\Delta\phi(j_1,\ell_1)$',
        'dphi_j2_l1': r'$\Delta\phi(j_2,\ell_1)$',
        'dphi_met_l1': r'$\Delta\phi(E_{\mathrm{T}}^{\mathrm{miss}},\ell_1)$',
        'dphi_jj': r'$\Delta\phi_{jj}$',
        'met_phi': r'$\phi_{\mathrm{miss}}\,[\mathrm{rad}]$',
        'zstar_l1': r'$Z_{\ell_1}^{\star}$',
        'zstar_l2': r'$Z_{\ell_2}^{\star}$',
        'ptprod_ll_over_jj': r'$\frac{p_{\mathrm{T}}^{\ell_1}p_{\mathrm{T}}^{\ell_2}}{p_{\mathrm{T}}^{j_1}p_{\mathrm{T}}^{j_2}}$',
        'l1_flavor_code': r'$\mathrm{flavor}(\ell_1)$',
        'l2_flavor_code': r'$\mathrm{flavor}(\ell_2)$',
    }
    return label_map.get(col, rf'$\mathrm{{{col}}}$')


def legend_label(sample: str) -> str:
    label_map = {
        'WWjj_EW-LL-WW_cmf': r'$W^{\pm}_{\mathrm{L}}W^{\pm}_{\mathrm{L}}$-EW',
        'WWjj_EW-LT-WW_cmf': r'$W^{\pm}_{\mathrm{L}}W^{\pm}_{\mathrm{T}}$-EW',
        'WWjj_EW-TT-WW_cmf': r'$W^{\pm}_{\mathrm{T}}W^{\pm}_{\mathrm{T}}$-EW',
        'WWjj_EW': r'$W^{\pm}W^{\pm}$-EW',
        'WWjj_QCD': r'$W^{\pm}W^{\pm}$-QCD',
        'WZjj_EW': r'$W^{\pm}Z$-EW',
        'WZjj_QCD': r'$W^{\pm}Z$-QCD'
    }
    return label_map.get(sample, rf'$\mathrm{{{sample}}}$')


sample_tables = defaultdict(list)
for rec in records:
    sample_tables[rec['sample']].append(pq.read_table(rec['parquet']))

merged_by_sample = {sample: pa.concat_tables(tables) for sample, tables in sample_tables.items()}
sample_names = sorted(merged_by_sample.keys())
print('Samples:', sample_names)

numeric_cols = [
    name for name in merged_by_sample[sample_names[0]].column_names
    if pa.types.is_integer(merged_by_sample[sample_names[0]].schema.field(name).type)
    or pa.types.is_floating(merged_by_sample[sample_names[0]].schema.field(name).type)
]

saved_overlay = []
for col in numeric_cols:
    per_sample_values = {}
    merged_values = []

    for sample in sample_names:
        arr = merged_by_sample[sample][col].to_numpy(zero_copy_only=False)
        finite = arr[np.isfinite(arr)]
        per_sample_values[sample] = finite
        if finite.size > 0:
            merged_values.append(finite)

    if not merged_values:
        continue

    all_vals = np.concatenate(merged_values)
    vmin, vmax = np.percentile(all_vals, [1, 99])
    if np.isclose(vmin, vmax):
        vmin, vmax = float(all_vals.min()), float(all_vals.max())

    if np.isclose(vmin, vmax):
        vmax = vmin + 1.0

    bins = np.linspace(vmin, vmax, 21)

    fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)

    for sample in sample_names:
        vals = per_sample_values[sample]
        if vals.size == 0:
            continue
        in_range = vals[(vals >= vmin) & (vals <= vmax)]
        if in_range.size == 0:
            continue
        ax.hist(
            in_range,
            bins=bins,
            histtype='step',
            linewidth=1.8,
            density=True,
            label=legend_label(sample),
        )

    ax.set_xlabel(x_label_math(col), fontsize=12)
    ax.set_ylabel(r'$\mathrm{Normalized\ events}$', fontsize=12)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9, frameon=False)

    pdf_path = overlay_dir / f'overlay_{sanitize_filename(col)}.pdf'
    fig.savefig(pdf_path, format='pdf')
    plt.close(fig)
    saved_overlay.append(pdf_path)

print(f'[EW polar] Saved {len(saved_overlay)} overlay PDFs to: {overlay_dir}')
print('First 10 files:')
for p in saved_overlay[:10]:
    print(' -', p)
if len(saved_overlay) > 10:
    print(f' ... and {len(saved_overlay) - 10} more files.')

Samples: ['WWjj_EW-LL-WW_cmf', 'WWjj_EW-LT-WW_cmf', 'WWjj_EW-TT-WW_cmf']
[EW polar] Saved 32 overlay PDFs to: /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar
First 10 files:
 - /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar/overlay_l1_pt.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar/overlay_l1_eta.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar/overlay_l1_flavor_code.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar/overlay_l2_pt.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar/overlay_l2_eta.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar/overlay_l2_flavor_code.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar/overlay_j1_pt.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar/overlay_j1_eta.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar/overlay_j2_pt.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_ew_polar/overlay_j2_e

In [ ]:
import json
import subprocess
from pathlib import Path

repo_root = Path('/home/r10222035/longitudinal_W')
script_path = repo_root / 'Sample/selection/export_sr_to_parquet.py'

target_dirs = [
    repo_root / 'Sample/MG5/WWjj_EW',
    repo_root / 'Sample/MG5/WWjj_QCD',
    repo_root / 'Sample/MG5/WZjj_EW',
    repo_root / 'Sample/MG5/WZjj_QCD',
]

# 先只測試 run_01；確認流程沒問題後再把 run_02 加進來。
target_runs = {'run_01', 'run_02'}

batch_out_dir = repo_root / 'Sample/Parquet/batch_sr_parquet_wwzz_mix'
batch_out_dir.mkdir(parents=True, exist_ok=True)
manifest_path = batch_out_dir / 'batch_manifest_wwzz_mix.json'

all_root_files = []
for d in target_dirs:
    roots = sorted(
        p for p in d.rglob('*.root')
        if any(run in p.parts for run in target_runs)
    )
    all_root_files.extend(roots)

assert script_path.exists(), f'Missing script: {script_path}'
assert all_root_files, 'No ROOT files found for selected runs in target directories.'

print(f'[WW/WZ mix] Runs: {sorted(target_runs)}')
print(f'[WW/WZ mix] Found {len(all_root_files)} ROOT files.')

converted_records = []
for i, root_file in enumerate(all_root_files, start=1):
    # /home/r10222035/longitudinal_W/Sample/MG5/<sample>/Events/<run>/xxx.root
    sample = root_file.parts[-4] if len(root_file.parts) >= 4 else root_file.parent.name
    run_name = root_file.parent.name

    out_name = f'{sample}_{run_name}_sr.parquet'
    out_parquet = batch_out_dir / out_name

    cmd = [
        'conda', 'run', '-n', 'vbs_wl',
        'python', str(script_path),
        '--input-root', str(root_file),
        '--tree', 'Delphes',
        '--output-parquet', str(out_parquet),
    ]

    print(f'[WW/WZ mix {i}/{len(all_root_files)}] sample={sample} converting: {root_file}')
    print(' '.join(cmd))
    proc = subprocess.run(cmd, cwd=str(repo_root))

    if proc.returncode != 0:
        raise RuntimeError(f'Conversion failed for {root_file}')

    cutflow_path = Path(str(out_parquet) + '.cutflow.json')
    assert out_parquet.exists(), f'Missing parquet: {out_parquet}'
    assert cutflow_path.exists(), f'Missing cutflow: {cutflow_path}'

    with cutflow_path.open('r', encoding='utf-8') as f:
        info = json.load(f)

    converted_records.append(
        {
            'sample': sample,
            'run': run_name,
            'input_root': str(root_file),
            'parquet': str(out_parquet),
            'cutflow_json': str(cutflow_path),
            'n_output_rows': int(info.get('n_output_rows', 0)),
            'cutflow': info.get('cutflow', {}),
        }
    )

manifest_path.write_text(json.dumps(converted_records, indent=2), encoding='utf-8')

print(f'[WW/WZ mix] Converted {len(converted_records)} files.')
print(f'[WW/WZ mix] Manifest: {manifest_path}')
for rec in converted_records:
    print(f"- {rec['sample']} {rec['run']}: rows={rec['n_output_rows']}")

In [2]:
import re
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq

repo_root = Path('/home/r10222035/longitudinal_W')
batch_out_dir = repo_root / 'Sample/Parquet/batch_sr_parquet_wwzz_mix'
manifest_path = batch_out_dir / 'batch_manifest_wwzz_mix.json'
overlay_dir = repo_root / 'figures' / 'sr_overlay_wwzz_mix'
overlay_dir.mkdir(parents=True, exist_ok=True)

assert manifest_path.exists(), f'Missing manifest: {manifest_path}'
records = json.loads(manifest_path.read_text(encoding='utf-8'))
assert records, 'Manifest is empty.'


def sanitize_filename(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', name)


def x_label_math(col: str) -> str:
    label_map = {
        'l1_pt': r'$p_{\mathrm{T}}^{\ell_1}\,[\mathrm{GeV}]$',
        'l2_pt': r'$p_{\mathrm{T}}^{\ell_2}\,[\mathrm{GeV}]$',
        'j1_pt': r'$p_{\mathrm{T}}^{j_1}\,[\mathrm{GeV}]$',
        'j2_pt': r'$p_{\mathrm{T}}^{j_2}\,[\mathrm{GeV}]$',
        'pt_ll': r'$p_{\mathrm{T}}^{\ell\ell}\,[\mathrm{GeV}]$',
        'met_et': r'$E_{\mathrm{T}}^{\mathrm{miss}}\,[\mathrm{GeV}]$',
        'm_ll': r'$m_{\ell\ell}\,[\mathrm{GeV}]$',
        'm_jj': r'$m_{jj}\,[\mathrm{GeV}]$',
        'mt_l1_met': r'$m_{\mathrm{T}}(\ell_1, E_{\mathrm{T}}^{\mathrm{miss}})\,[\mathrm{GeV}]$',
        'mt_l2_met': r'$m_{\mathrm{T}}(\ell_2, E_{\mathrm{T}}^{\mathrm{miss}})\,[\mathrm{GeV}]$',
        'mt_ll_met': r'$m_{\mathrm{T}}(\ell\ell, E_{\mathrm{T}}^{\mathrm{miss}})\,[\mathrm{GeV}]$',
        'mt0_ll_met': r'$m_{\mathrm{T}}^{0}(\ell\ell, E_{\mathrm{T}}^{\mathrm{miss}})\,[\mathrm{GeV}]$',
        'l1_eta': r'$\eta_{\ell_1}$',
        'l2_eta': r'$\eta_{\ell_2}$',
        'j1_eta': r'$\eta_{j_1}$',
        'j2_eta': r'$\eta_{j_2}$',
        'deta_ll': r'$\Delta\eta_{\ell\ell}$',
        'dy_jj': r'$\Delta y_{jj}$',
        'dr_ll': r'$\Delta R_{\ell\ell}$',
        'dr_jj': r'$\Delta R_{jj}$',
        'min_dr_lj': r'$\min(\Delta R_{\ell,j})$',
        'dphi_l2_l1': r'$\Delta\phi(\ell_2,\ell_1)$',
        'dphi_j1_l1': r'$\Delta\phi(j_1,\ell_1)$',
        'dphi_j2_l1': r'$\Delta\phi(j_2,\ell_1)$',
        'dphi_met_l1': r'$\Delta\phi(E_{\mathrm{T}}^{\mathrm{miss}},\ell_1)$',
        'dphi_jj': r'$\Delta\phi_{jj}$',
        'met_phi': r'$\phi_{\mathrm{miss}}\,[\mathrm{rad}]$',
        'zstar_l1': r'$Z_{\ell_1}^{\star}$',
        'zstar_l2': r'$Z_{\ell_2}^{\star}$',
        'ptprod_ll_over_jj': r'$\frac{p_{\mathrm{T}}^{\ell_1}p_{\mathrm{T}}^{\ell_2}}{p_{\mathrm{T}}^{j_1}p_{\mathrm{T}}^{j_2}}$',
        'l1_flavor_code': r'$\mathrm{flavor}(\ell_1)$',
        'l2_flavor_code': r'$\mathrm{flavor}(\ell_2)$',
    }
    return label_map.get(col, rf'$\mathrm{{{col}}}$')


def legend_label(sample: str) -> str:
    label_map = {
        'WWjj_EW-LL-WW_cmf': r'$W^{\pm}_{\mathrm{L}}W^{\pm}_{\mathrm{L}}$-EW',
        'WWjj_EW-LT-WW_cmf': r'$W^{\pm}_{\mathrm{L}}W^{\pm}_{\mathrm{T}}$-EW',
        'WWjj_EW-TT-WW_cmf': r'$W^{\pm}_{\mathrm{T}}W^{\pm}_{\mathrm{T}}$-EW',
        'WWjj_EW': r'$W^{\pm}W^{\pm}$-EW',
        'WWjj_QCD': r'$W^{\pm}W^{\pm}$-QCD',
        'WZjj_EW': r'$W^{\pm}Z$-EW',
        'WZjj_QCD': r'$W^{\pm}Z$-QCD'
    }
    return label_map.get(sample, rf'$\mathrm{{{sample}}}$')


sample_tables = defaultdict(list)
for rec in records:
    sample_tables[rec['sample']].append(pq.read_table(rec['parquet']))

merged_by_sample = {sample: pa.concat_tables(tables) for sample, tables in sample_tables.items()}
sample_names = sorted(merged_by_sample.keys())
print('Samples:', sample_names)

numeric_cols = [
    name for name in merged_by_sample[sample_names[0]].column_names
    if pa.types.is_integer(merged_by_sample[sample_names[0]].schema.field(name).type)
    or pa.types.is_floating(merged_by_sample[sample_names[0]].schema.field(name).type)
]

saved_overlay = []
for col in numeric_cols:
    per_sample_values = {}
    merged_values = []

    for sample in sample_names:
        arr = merged_by_sample[sample][col].to_numpy(zero_copy_only=False)
        finite = arr[np.isfinite(arr)]
        per_sample_values[sample] = finite
        if finite.size > 0:
            merged_values.append(finite)

    if not merged_values:
        continue

    all_vals = np.concatenate(merged_values)
    vmin, vmax = np.percentile(all_vals, [1, 99])
    if np.isclose(vmin, vmax):
        vmin, vmax = float(all_vals.min()), float(all_vals.max())

    if np.isclose(vmin, vmax):
        vmax = vmin + 1.0

    bins = np.linspace(vmin, vmax, 21)

    fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)

    for sample in sample_names:
        vals = per_sample_values[sample]
        if vals.size == 0:
            continue
        in_range = vals[(vals >= vmin) & (vals <= vmax)]
        if in_range.size == 0:
            continue
        ax.hist(
            in_range,
            bins=bins,
            histtype='step',
            linewidth=1.8,
            density=True,
            label=legend_label(sample),
        )

    ax.set_xlabel(x_label_math(col), fontsize=12)
    ax.set_ylabel(r'$\mathrm{Normalized\ events}$', fontsize=12)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9, frameon=False)

    pdf_path = overlay_dir / f'overlay_{sanitize_filename(col)}.pdf'
    fig.savefig(pdf_path, format='pdf')
    plt.close(fig)
    saved_overlay.append(pdf_path)

print(f'[WW/WZ mix] Saved {len(saved_overlay)} overlay PDFs to: {overlay_dir}')
print('First 10 files:')
for p in saved_overlay[:10]:
    print(' -', p)
if len(saved_overlay) > 10:
    print(f' ... and {len(saved_overlay) - 10} more files.')

Samples: ['WWjj_EW', 'WWjj_QCD', 'WZjj_EW', 'WZjj_QCD']
[WW/WZ mix] Saved 32 overlay PDFs to: /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix
First 10 files:
 - /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix/overlay_l1_pt.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix/overlay_l1_eta.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix/overlay_l1_flavor_code.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix/overlay_l2_pt.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix/overlay_l2_eta.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix/overlay_l2_flavor_code.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix/overlay_j1_pt.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix/overlay_j1_eta.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix/overlay_j2_pt.pdf
 - /home/r10222035/longitudinal_W/figures/sr_overlay_wwzz_mix/overlay_j2_eta.pdf
 ... and 